# LeetCode 484 — Find Permutation bằng Python

Link đề: https://leetcode.com/problems/find-permutation/ · Độ khó: **Medium** · Chủ đề: Stack, Greedy, Array, String, Hoán vị và chuỗi `I`/`D`

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def findPermutation(self, s: str) -> List[int]:
        ...
```

Ghi chú: đây là bài **Premium** của LeetCode. Đề trong notebook được viết lại theo hiểu biết về bài, nên hãy đối chiếu lại đề gốc nếu bạn có tài khoản Premium. Kết quả của bài là **duy nhất** (hoán vị nhỏ nhất theo thứ tự từ điển), nhưng để chắc chắn, notebook test bằng một **hàm kiểm tra tính hợp lệ** cộng với **oracle vét cạn** (duyệt mọi hoán vị theo thứ tự từ điển, lấy hoán vị hợp lệ đầu tiên).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một hoán vị của `1..n+1` được "mã hóa" thành chuỗi `I`/`D` độ dài `n` (`I` = tăng, `D` = giảm giữa hai phần tử liền kề); cho chuỗi, phải dựng lại hoán vị **nhỏ nhất theo thứ tự từ điển** (lexicographically smallest).
2. Dùng được các công cụ Python: `range`, `list(range(...))`, cắt lát và **đảo ngược đoạn** `a[i:j+1] = a[i:j+1][::-1]`, `itertools.permutations`, danh sách làm **ngăn xếp** (`append`/`pop`), so sánh danh sách theo từ điển.
3. Nắm hai trực giác: (a) bắt đầu từ dãy tăng `1, 2, ..., n+1` rồi **đảo ngược mỗi đoạn có `D`**; (b) ngăn xếp "dồn số vào khi gặp `D`, đổ ra khi gặp `I`".
4. Cài đặt 5 cách khác nhau về ý tưởng: **duyệt mọi hoán vị → quay lui theo thứ tự từ điển → đảo ngược các đoạn `D` → ngăn xếp → công thức dùng mảng đếm chuỗi `D` từ phải**, và so sánh độ phức tạp.
5. Tránh các lỗi kinh điển: sai độ dài kết quả (`n+1`), đảo thiếu một phần tử ở cuối đoạn, dùng cách tham lam kiểu "hoán vị bất kỳ" (hợp lệ nhưng **không nhỏ nhất**), quên đổ ngăn xếp ở cuối.

## 2. Đề bài & ví dụ

Một hoán vị `perm` gồm **tất cả** các số nguyên trong đoạn `[1, n+1]` có thể được biểu diễn bằng một chuỗi `s` độ dài `n` như sau:

- `s[i] == 'I'` nếu `perm[i] < perm[i + 1]` (tăng),
- `s[i] == 'D'` nếu `perm[i] > perm[i + 1]` (giảm).

Cho chuỗi `s`, hãy dựng lại hoán vị `perm` **nhỏ nhất theo thứ tự từ điển** và trả về nó.

**Ví dụ 1:** `s = "I"` → `[1, 2]`. Chỉ có đúng một hoán vị `[1, 2]` có quan hệ tăng.

**Ví dụ 2:** `s = "DI"` → `[2, 1, 3]`. Cả `[2, 1, 3]` và `[3, 1, 2]` đều biểu diễn thành `"DI"`, nhưng ta cần hoán vị nhỏ nhất theo thứ tự từ điển nên chọn `[2, 1, 3]`.

**Ràng buộc:** `1 <= s.length <= 10^5`; `s[i]` là `'I'` hoặc `'D'`.

Chú ý: kết quả có độ dài **`n + 1`** (một hơn độ dài chuỗi `s`) và vì yêu cầu "nhỏ nhất theo thứ tự từ điển" nên đáp án là **duy nhất**.

In [1]:
from itertools import permutations
from typing import List

def ma_hoa(perm):
    """Từ hoán vị suy ra chuỗi I/D (chiều ngược lại của đề bài)."""
    return "".join("I" if perm[i] < perm[i + 1] else "D" for i in range(len(perm) - 1))

print(ma_hoa([1, 2]))          # I
print(ma_hoa([2, 1, 3]))       # DI
print(ma_hoa([3, 1, 2]))       # DI  -> cùng một chuỗi, nhiều hoán vị

I
DI
DI


## 3. Kiến thức nền

### 3.1. `range`, `list(range(...))` để dựng dãy `1..n+1`

`range(a, b)` đi từ `a` đến `b - 1`. Muốn có các số `1, 2, ..., n+1` ta dùng `range(1, n + 2)`; bọc `list(...)` để có danh sách thật.

In [2]:
n = 4
print(list(range(1, n + 2)))      # [1, 2, 3, 4, 5]  -> n + 1 = 5 số
print(list(range(5, 0, -1)))      # [5, 4, 3, 2, 1]  -> bước -1 để đếm lùi

[1, 2, 3, 4, 5]
[5, 4, 3, 2, 1]


### 3.2. Cắt lát và đảo ngược một đoạn trong danh sách

`a[l : r + 1]` lấy các phần tử từ chỉ số `l` đến `r` (cả hai đầu). `[::-1]` đảo ngược một danh sách. Gán lại `a[l : r + 1] = a[l : r + 1][::-1]` đảo ngược **đúng đoạn đó tại chỗ**; các phần tử khác không đổi.

In [3]:
a = [1, 2, 3, 4, 5]
print(a[1:4])                 # [2, 3, 4]
a[1:4] = a[1:4][::-1]         # đảo đoạn chỉ số 1..3
print(a)                      # [1, 4, 3, 2, 5]

[2, 3, 4]
[1, 4, 3, 2, 5]


### 3.3. Ngăn xếp (stack) bằng danh sách

Ngăn xếp là cấu trúc "vào sau, ra trước" (LIFO): ta chỉ **thêm** ở cuối (`append`) và **lấy** từ cuối (`pop`). Hãy tưởng tượng chồng đĩa: đĩa đặt sau cùng được lấy ra trước. Nhờ tính chất này, khi **đổ** ngăn xếp ra, thứ tự bị **đảo ngược**: đẩy `1, 2, 3` vào rồi lấy ra được `3, 2, 1`. Đó chính là thứ ta cần để tạo đoạn giảm!

In [4]:
st = []
for x in [1, 2, 3]:
    st.append(x)
print(st)                      # [1, 2, 3]
out = []
while st:
    out.append(st.pop())
print(out)                     # [3, 2, 1]  -> đảo ngược

[1, 2, 3]
[3, 2, 1]


### 3.4. `itertools.permutations` và thứ tự từ điển

`permutations(iterable)` sinh các hoán vị. Nếu phần tử đầu vào **đã sắp tăng**, các hoán vị sinh ra theo **thứ tự từ điển** (so sánh phần tử đầu, hòa thì so phần tử kế tiếp...). Python so sánh hai danh sách/tuple đúng theo thứ tự từ điển, nên `(2, 1, 3) < (3, 1, 2)` là `True`.

In [5]:
for p in permutations([1, 2, 3]):
    print(p, end="  ")
print()
print((2, 1, 3) < (3, 1, 2), [1, 3, 2] < [2, 1, 3])      # True True

(1, 2, 3)  (1, 3, 2)  (2, 1, 3)  (2, 3, 1)  (3, 1, 2)  (3, 2, 1)  
True True


### 3.5. Đếm các ký tự `D` liên tiếp

Ta sẽ cần biết "từ vị trí `i` có bao nhiêu chữ `D` đứng liên tiếp". Quét **từ phải sang trái** là cách tự nhiên: nếu `s[i] == 'D'` thì `d[i] = d[i+1] + 1`, ngược lại `d[i] = 0`.

In [6]:
s = "DDIDI"
n = len(s)
d = [0] * (n + 1)                  # d[n] = 0 làm mốc
for i in range(n - 1, -1, -1):
    d[i] = d[i + 1] + 1 if s[i] == "D" else 0
print(s)
print(d)                           # [2, 1, 0, 1, 0, 0]

DDIDI
[2, 1, 0, 1, 0, 0]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

**Ví dụ `s = "DI"`.** Hoán vị có 3 phần tử, quan hệ: `p0 > p1 < p2`. Có hai hoán vị: `[2, 1, 3]` và `[3, 1, 2]`. So sánh từ điển, `[2,1,3]` nhỏ hơn.

**Ví dụ `s = "DDI"`** (4 phần tử, `p0 > p1 > p2 < p3`): `p0` có hai phần tử nhỏ hơn đứng ngay sau, nên `p0 ≥ 3`. Chọn `p0 = 3` thì `p1, p2` phải là hai số nhỏ hơn 3, giảm dần: `2, 1`. Còn lại `4` ở cuối: `[3, 2, 1, 4]`. Đó là nhỏ nhất.

**Ví dụ `s = "IDID"`** (5 phần tử): `p0 < p1 > p2 < p3 > p4`. Muốn `p0` nhỏ nhất ⇒ `p0 = 1`. Tiếp theo `p1` mở đầu một chuỗi giảm dài 2 (`p1 > p2`) ⇒ `p1 ≥ 3`: chọn `3`, rồi `p2 = 2`; `p3` mở chuỗi giảm `p3 > p4` ⇒ `p3 = 5`, `p4 = 4`. Kết quả `[1, 3, 2, 5, 4]`.

**Quan sát:** nếu bắt đầu từ `1, 2, ..., n+1` rồi **đảo ngược các đoạn tương ứng với chuỗi `D` liên tiếp** thì ta ra đúng các đáp án trên:

- `"DI"`: đoạn `D` ở vị trí 0 phủ các phần tử vị trí `0..1` → đảo `[1,2]` thành `[2,1]` → `[2, 1, 3]`.
- `"DDI"`: `D D` phủ vị trí `0..2` → đảo `[1,2,3]` → `[3,2,1,4]`.
- `"IDID"`: `D` ở s[1] phủ vị trí `1..2` → đảo `[2,3]` → `[1,3,2,4,5]`; `D` ở s[3] phủ vị trí `3..4` → đảo `[4,5]` → `[1,3,2,5,4]`.

In [7]:
def minh_hoa_dao(s):
    a = list(range(1, len(s) + 2))
    print("bắt đầu:", a)
    i = 0
    while i < len(s):
        if s[i] == "D":
            j = i
            while j < len(s) and s[j] == "D":
                j += 1                        # a[i..j] là đoạn cần đảo
            a[i:j + 1] = a[i:j + 1][::-1]
            print(f"đảo đoạn [{i}..{j}] -> {a}")
            i = j
        else:
            i += 1
    return a

print(minh_hoa_dao("IDID"))
print(minh_hoa_dao("DDI"))

bắt đầu: [1, 2, 3, 4, 5]
đảo đoạn [1..2] -> [1, 3, 2, 4, 5]
đảo đoạn [3..4] -> [1, 3, 2, 5, 4]
[1, 3, 2, 5, 4]
bắt đầu: [1, 2, 3, 4]
đảo đoạn [0..2] -> [3, 2, 1, 4]
[3, 2, 1, 4]


## 5. Cách 1 — Brute Force / baseline: duyệt mọi hoán vị theo thứ tự từ điển

**Ý tưởng:** bám sát đề. Sinh **tất cả** hoán vị của `1..n+1` theo **thứ tự từ điển**, hoán vị **đầu tiên** có chuỗi `I`/`D` bằng `s` chính là đáp án (vì ta duyệt từ nhỏ đến lớn).

Ta cần hàm kiểm tra tính hợp lệ độc lập (`hop_le`) để dùng làm "trọng tài" cho mọi cách sau.

In [8]:
def hop_le(s, perm):
    """perm có phải hoán vị của 1..len(s)+1 thỏa mọi quan hệ I/D của s không?"""
    n = len(s)
    if len(perm) != n + 1 or sorted(perm) != list(range(1, n + 2)):
        return False
    for k, ch in enumerate(s):
        if ch == "I" and not perm[k] < perm[k + 1]:
            return False
        if ch == "D" and not perm[k] > perm[k + 1]:
            return False
    return True

def vet_can(s):
    for p in permutations(range(1, len(s) + 2)):      # đã theo thứ tự từ điển
        if hop_le(s, p):
            return list(p)

print(vet_can("I"))       # [1, 2]
print(vet_can("DI"))      # [2, 1, 3]
print(vet_can("DDI"))     # [3, 2, 1, 4]
print(vet_can("IDID"))    # [1, 3, 2, 5, 4]

[1, 2]
[2, 1, 3]
[3, 2, 1, 4]
[1, 3, 2, 5, 4]


**Độ phức tạp:** có `(n+1)!` hoán vị, mỗi cái kiểm tra `O(n)` ⇒ thời gian `O(n · (n+1)!)` (xấu nhất), bộ nhớ `O(n)`. Chỉ dùng được với `n ≲ 8`. Hãy xem nó tăng đến mức nào:

In [9]:
from math import factorial
for n in (3, 5, 8, 10, 15, 20):
    print(f"n = {n:>2}: (n+1)! = {factorial(n + 1):>22,}")

n =  3: (n+1)! =                     24
n =  5: (n+1)! =                    720
n =  8: (n+1)! =                362,880
n = 10: (n+1)! =             39,916,800
n = 15: (n+1)! =     20,922,789,888,000
n = 20: (n+1)! = 51,090,942,171,709,440,000


Với `n = 10^5` thì con số này vượt xa mọi tưởng tượng. Ta cần một cách **dựng thẳng** đáp án, không thử.

## 6. Ý tưởng thuật toán / bước đột phá

1. **Chia chuỗi thành các "khối"** bằng cách cắt sau mỗi chữ `I`. Mỗi khối có dạng `D...D` rồi một `I` (hoặc kết thúc chuỗi). Khối phủ các vị trí hoán vị `l..r` với `r - l` chữ `D` đầu khối.
2. **Trong khối, giá trị phải giảm dần;** giữa hai khối kề nhau, phần tử cuối khối trước **nhỏ hơn** phần tử đầu khối sau (vì có chữ `I`).
3. **Để nhỏ nhất theo từ điển,** mỗi khối nên dùng đúng các số **nhỏ nhất còn lại**, sắp giảm dần: khối `l..r` dùng `l+1, ..., r+1` theo thứ tự `r+1, r, ..., l+1`.
4. Điều này giống hệt "bắt đầu từ `1..n+1` rồi **đảo ngược từng đoạn** có chữ `D`".
5. Khi khối kế tiếp dùng các số **lớn hơn** mọi số của khối trước, quan hệ `I` giữa hai khối tự thỏa.
6. Không cần thử bất cứ khả năng nào khác ⇒ thời gian tuyến tính `O(n)`.

## 7. Các cách tiếp theo

### Cách 2 — Quay lui (backtracking) theo thứ tự từ điển, dừng ở nghiệm đầu tiên

**Ý tưởng khác:** thay vì sinh **mọi** hoán vị rồi lọc, ta **xây từng phần tử**: ở vị trí `k` thử các giá trị chưa dùng **từ nhỏ đến lớn**, chỉ nhận giá trị thỏa quan hệ với phần tử trước (`I`: lớn hơn, `D`: nhỏ hơn); đệ quy tiếp; nếu bế tắc thì lùi lại thử giá trị khác. Vì thử theo thứ tự tăng, **nghiệm đầu tiên tìm được** chính là nhỏ nhất theo từ điển. Cách này cắt tỉa sớm nhưng trường hợp xấu vẫn rất chậm.

**Giải thích từng đoạn:**

- `used[v]` đánh dấu số `v` đã dùng; `perm` là hoán vị đang dựng.
- `dfs()` trả `True` khi `len(perm) == n + 1` (đã đủ).
- Với vị trí `k = len(perm)`: duyệt `v = 1..n+1`; bỏ qua `v` đã dùng; nếu `k > 0` kiểm tra quan hệ giữa `perm[-1]` và `v` theo `s[k-1]`.
- Chọn `v`, đệ quy; nếu thất bại thì bỏ chọn (`pop`, `used[v] = False`).
- Đếm số lần gọi để thấy nó bùng nổ.

In [10]:
def quay_lui(s, dem_goi=None):
    n1 = len(s) + 1
    used = [False] * (n1 + 1)
    perm = []

    def dfs():
        if dem_goi is not None:
            dem_goi[0] += 1
        if len(perm) == n1:
            return True
        k = len(perm)
        for v in range(1, n1 + 1):
            if used[v]:
                continue
            if k > 0:
                if s[k - 1] == "I" and not perm[-1] < v:
                    continue
                if s[k - 1] == "D" and not perm[-1] > v:
                    continue
            used[v] = True
            perm.append(v)
            if dfs():
                return True
            perm.pop()
            used[v] = False
        return False

    dfs()
    return perm

print(quay_lui("DI"), quay_lui("DDI"), quay_lui("IDID"))
for n in (4, 6, 8, 9):
    c = [0]
    quay_lui("D" * n, c)
    print(f"s = 'D'*{n}: số lần gọi dfs = {c[0]}")

[2, 1, 3] [3, 2, 1, 4] [1, 3, 2, 5, 4]
s = 'D'*4: số lần gọi dfs = 32
s = 'D'*6: số lần gọi dfs = 128
s = 'D'*8: số lần gọi dfs = 512
s = 'D'*9: số lần gọi dfs = 1024


**Nhận xét:** với `"D" * n` (dãy giảm hoàn toàn) quay lui phải thử gần hết khả năng rồi mới nhận ra đáp án là `n+1, n, ..., 1`, số lần gọi tăng mạnh theo `n`. Hơn nữa, đệ quy sâu `n+1` tầng sẽ vượt giới hạn đệ quy của Python khi `n = 10^5`.

**Độ phức tạp:** xấu nhất cỡ hàm mũ/giai thừa `O((n+1)!)`, bộ nhớ `O(n)`. Không dùng để nộp.

### Cách 3 — Đảo ngược các đoạn `D` (hai con trỏ, tại chỗ)

**Giải thích từng đoạn:**

- `ans = list(range(1, n + 2))`: dãy tăng `1..n+1` (độ dài `n+1`).
- `i` là đầu khối hiện tại. Nếu `s[i] == 'D'`, đặt `j = i` rồi tăng `j` **khi `s[j] == 'D'`** để tìm hết chuỗi `D` liên tiếp. Khi dừng, các vị trí `i..j` (cả `j`) thuộc cùng một khối giảm.
- Đảo đoạn `ans[i : j + 1]`.
- Nhảy `i = max(i + 1, j)`: nếu không có `D` (`j == i`) thì tiến 1; nếu có thì nhảy tới `j` (phần tử cuối khối).

In [11]:
def dao_doan(s):
    n = len(s)
    ans = list(range(1, n + 2))
    i = 0
    while i < n:
        j = i
        while j < n and s[j] == "D":
            j += 1
        ans[i:j + 1] = ans[i:j + 1][::-1]
        i = max(i + 1, j)
    return ans

for s in ("I", "DI", "DDI", "IDID", "DDDD", "IIII", "DIDI"):
    kq = dao_doan(s)
    print(f"{s!r:8} -> {kq}")
    assert kq == vet_can(s)

'I'      -> [1, 2]
'DI'     -> [2, 1, 3]
'DDI'    -> [3, 2, 1, 4]
'IDID'   -> [1, 3, 2, 5, 4]
'DDDD'   -> [5, 4, 3, 2, 1]
'IIII'   -> [1, 2, 3, 4, 5]
'DIDI'   -> [2, 1, 4, 3, 5]


**Code nộp LeetCode (Cách 3):**

```python
from typing import List

class Solution:
    def findPermutation(self, s: str) -> List[int]:
        n = len(s)
        ans = list(range(1, n + 2))
        i = 0
        while i < n:
            j = i
            while j < n and s[j] == 'D':
                j += 1
            ans[i:j + 1] = ans[i:j + 1][::-1]
            i = max(i + 1, j)
        return ans
```

**Độ phức tạp:** mỗi chữ `D` bị quét đúng một lần và mỗi đoạn đảo một lần ⇒ thời gian `O(n)`; bộ nhớ `O(1)` thêm (ngoài kết quả `O(n)`).

### Cách 4 — Ngăn xếp: dồn số vào, gặp `I` (hoặc hết chuỗi) thì đổ ra

**Ý tưởng khác:** thay vì đảo, ta **đẩy lần lượt** `1, 2, ..., n+1` vào ngăn xếp. Mỗi khi `s[i-1] == 'I'` (giữa `i` và `i+1` là tăng, nghĩa là khối hiện tại kết thúc tại `i`), hoặc đã hết số, ta **đổ toàn bộ ngăn xếp ra** kết quả (đảo ngược nhờ LIFO). Gặp `D` thì cứ tiếp tục dồn.

**Giải thích từng đoạn:**

- Với `i` chạy `1 .. n+1`: `st.append(i)`.
- Nếu `i == n + 1` (số cuối) hoặc `s[i - 1] == 'I'` (giữa `i` và `i+1` là tăng): đổ ngăn xếp: `while st: res.append(st.pop())`.
- Khi `s[i-1] == 'D'` thì `i` và `i+1` phải giảm, nên chưa đổ; `i` nằm lại trong ngăn xếp để sau này ra **sau** `i+1`.

In [12]:
def ngan_xep(s):
    n = len(s)
    res, st = [], []
    for i in range(1, n + 2):
        st.append(i)
        if i == n + 1 or s[i - 1] == "I":
            while st:
                res.append(st.pop())
    return res

for s in ("I", "DI", "DDI", "IDID", "DDDD", "IIII", "DIDI"):
    kq = ngan_xep(s)
    print(f"{s!r:8} -> {kq}")
    assert kq == vet_can(s)

'I'      -> [1, 2]
'DI'     -> [2, 1, 3]
'DDI'    -> [3, 2, 1, 4]
'IDID'   -> [1, 3, 2, 5, 4]
'DDDD'   -> [5, 4, 3, 2, 1]
'IIII'   -> [1, 2, 3, 4, 5]
'DIDI'   -> [2, 1, 4, 3, 5]


**Code nộp LeetCode (Cách 4):**

```python
from typing import List

class Solution:
    def findPermutation(self, s: str) -> List[int]:
        n = len(s)
        res, st = [], []
        for i in range(1, n + 2):
            st.append(i)
            if i == n + 1 or s[i - 1] == 'I':
                while st:
                    res.append(st.pop())
        return res
```

**Độ phức tạp:** mỗi số vào/ra ngăn xếp đúng một lần ⇒ thời gian `O(n)`, bộ nhớ `O(n)` cho ngăn xếp (trường hợp `"D"*n`).

### Cách 5 — Công thức: mảng `d` (số `D` liên tiếp tính từ phải) + một lượt trái sang phải

**Ý tưởng khác:** không đảo, không ngăn xếp: tính **trực tiếp** giá trị từng vị trí. Gọi `d[i]` là số chữ `D` liên tiếp bắt đầu tại `s[i]` (mục 3.5, `d[n] = 0`). Với vị trí `i` của hoán vị:

- Nếu `i > 0` và `s[i-1] == 'D'` thì `i` nằm **giữa/cuối khối giảm**: `ans[i] = ans[i-1] - 1`.
- Ngược lại `i` là **đầu khối**: khối này phủ `d[i] + 1` vị trí và dùng các số `i+1, ..., i+1+d[i]`, phần tử đầu là số lớn nhất: `ans[i] = i + 1 + d[i]`.

**Giải thích từng đoạn:** (1) quét phải→trái dựng `d`; (2) quét trái→phải điền `ans` theo hai trường hợp trên.

In [13]:
def cong_thuc(s):
    n = len(s)
    d = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        d[i] = d[i + 1] + 1 if s[i] == "D" else 0
    ans = [0] * (n + 1)
    for i in range(n + 1):
        if i > 0 and s[i - 1] == "D":
            ans[i] = ans[i - 1] - 1
        else:
            ans[i] = i + 1 + d[i]
    return ans

for s in ("I", "DI", "DDI", "IDID", "DDDD", "IIII", "DIDI"):
    kq = cong_thuc(s)
    print(f"{s!r:8} -> {kq}")
    assert kq == vet_can(s)

'I'      -> [1, 2]
'DI'     -> [2, 1, 3]
'DDI'    -> [3, 2, 1, 4]
'IDID'   -> [1, 3, 2, 5, 4]
'DDDD'   -> [5, 4, 3, 2, 1]
'IIII'   -> [1, 2, 3, 4, 5]
'DIDI'   -> [2, 1, 4, 3, 5]


**Code nộp LeetCode (Cách 5):**

```python
from typing import List

class Solution:
    def findPermutation(self, s: str) -> List[int]:
        n = len(s)
        d = [0] * (n + 1)
        for i in range(n - 1, -1, -1):
            d[i] = d[i + 1] + 1 if s[i] == 'D' else 0
        ans = [0] * (n + 1)
        for i in range(n + 1):
            if i > 0 and s[i - 1] == 'D':
                ans[i] = ans[i - 1] - 1
            else:
                ans[i] = i + 1 + d[i]
        return ans
```

**Độ phức tạp:** hai lượt quét ⇒ thời gian `O(n)`, bộ nhớ `O(n)` cho `d` (và kết quả).

## 8. TRACE từng bước

Dùng `s = "DDIDI"` (`n = 5`, hoán vị có 6 phần tử). Đáp án mong đợi `[3, 2, 1, 5, 4, 6]`.

**Cách 3 (đảo đoạn):**

| i | s[i] | j sau khi quét D | đoạn đảo | ans sau |
|---|---|---|---|---|
| 0 | D | 2 | ans[0..2] | 3 2 1 4 5 6 |
| 2 | I | 2 | ans[2..2] (không đổi) | 3 2 1 4 5 6 |
| 3 | D | 4 | ans[3..4] | 3 2 1 5 4 6 |
| 4 | I | 4 | ans[4..4] (không đổi) | 3 2 1 5 4 6 |
| kết thúc | | | `i = 5 = n` | **3 2 1 5 4 6** |

In [14]:
def trace_dao_doan(s):
    n = len(s)
    ans = list(range(1, n + 2))
    print("s =", s, "| ans ban đầu =", ans)
    i = 0
    while i < n:
        j = i
        while j < n and s[j] == "D":
            j += 1
        truoc = ans[:]
        ans[i:j + 1] = ans[i:j + 1][::-1]
        print(f"i={i}, s[i]={s[i]}, j={j}, đảo ans[{i}..{j}]: {truoc} -> {ans}")
        i = max(i + 1, j)
    print("kết quả:", ans)
    return ans

assert trace_dao_doan("DDIDI") == [3, 2, 1, 5, 4, 6]

s = DDIDI | ans ban đầu = [1, 2, 3, 4, 5, 6]
i=0, s[i]=D, j=2, đảo ans[0..2]: [1, 2, 3, 4, 5, 6] -> [3, 2, 1, 4, 5, 6]
i=2, s[i]=I, j=2, đảo ans[2..2]: [3, 2, 1, 4, 5, 6] -> [3, 2, 1, 4, 5, 6]
i=3, s[i]=D, j=4, đảo ans[3..4]: [3, 2, 1, 4, 5, 6] -> [3, 2, 1, 5, 4, 6]
i=4, s[i]=I, j=4, đảo ans[4..4]: [3, 2, 1, 5, 4, 6] -> [3, 2, 1, 5, 4, 6]
kết quả: [3, 2, 1, 5, 4, 6]


**Cách 4 (ngăn xếp)** cùng `s = "DDIDI"`:

| i | thêm vào st | s[i-1] | đổ? | st sau | res |
|---|---|---|---|---|---|
| 1 | 1 | D | không | [1] | [] |
| 2 | 2 | D | không | [1,2] | [] |
| 3 | 3 | I | **đổ** | [] | [3,2,1] |
| 4 | 4 | D | không | [4] | [3,2,1] |
| 5 | 5 | I | **đổ** | [] | [3,2,1,5,4] |
| 6 | 6 | (hết) | **đổ** | [] | [3,2,1,5,4,6] |

In [15]:
def trace_ngan_xep(s):
    n = len(s)
    res, st = [], []
    for i in range(1, n + 2):
        st.append(i)
        dau = s[i - 1] if i <= n else "(hết)"
        do = (i == n + 1 or s[i - 1] == "I")
        print(f"i={i}: đẩy {i} -> st={st}, s[i-1]={dau}, đổ? {do}", end="")
        if do:
            while st:
                res.append(st.pop())
        print(f" | res={res}")
    return res

assert trace_ngan_xep("DDIDI") == [3, 2, 1, 5, 4, 6]

i=1: đẩy 1 -> st=[1], s[i-1]=D, đổ? False | res=[]
i=2: đẩy 2 -> st=[1, 2], s[i-1]=D, đổ? False | res=[]
i=3: đẩy 3 -> st=[1, 2, 3], s[i-1]=I, đổ? True | res=[3, 2, 1]
i=4: đẩy 4 -> st=[4], s[i-1]=D, đổ? False | res=[3, 2, 1]
i=5: đẩy 5 -> st=[4, 5], s[i-1]=I, đổ? True | res=[3, 2, 1, 5, 4]
i=6: đẩy 6 -> st=[6], s[i-1]=(hết), đổ? True | res=[3, 2, 1, 5, 4, 6]


## 9. Code nộp LeetCode

Bản dùng Cách 3 (đảo đoạn), đúng `class Solution` và chữ ký:

In [16]:
from typing import List

class Solution:
    def findPermutation(self, s: str) -> List[int]:
        n = len(s)
        ans = list(range(1, n + 2))
        i = 0
        while i < n:
            j = i
            while j < n and s[j] == 'D':
                j += 1
            ans[i:j + 1] = ans[i:j + 1][::-1]
            i = max(i + 1, j)
        return ans

sol = Solution()
print(sol.findPermutation("I"), sol.findPermutation("DI"))
assert sol.findPermutation("I") == [1, 2]
assert sol.findPermutation("DI") == [2, 1, 3]

[1, 2] [2, 1, 3]


## 10. Vì sao thuật toán đúng

Chia chuỗi thành các **khối**: khối kết thúc ngay sau mỗi chữ `I` (hoặc ở cuối chuỗi). Khối phủ các vị trí hoán vị `l..r` với `s[l..r-1]` toàn `D` và `s[r] == 'I'` (hoặc `r = n`).

**Hợp lệ.** Đáp án điền khối `l..r` bằng các giá trị `r+1, r, ..., l+1` (giảm dần). Trong khối, mọi cặp liền kề đều giảm — khớp với `D`. Giữa khối này và khối sau: phần tử cuối khối này là `l+1`, phần tử đầu khối sau là `r'+1 ≥ r+2 > l+1` ⇒ tăng, khớp với `I`. Mọi giá trị `1..n+1` xuất hiện đúng một lần (các khối dùng các đoạn giá trị rời nhau, liên tiếp).

**Nhỏ nhất theo từ điển.** Lấy bất kỳ hoán vị hợp lệ `p`. Ta chứng minh `ans ≤ p` theo từng khối bằng quy nạp. Giả sử hai hoán vị trùng nhau trước vị trí `l` (đầu khối): khi đó `p` đã dùng đúng các số `1..l`, và các số còn lại là `l+1, l+2, ...`. Khối `l..r` có `k = r - l + 1` phần tử giảm dần, tất cả lấy từ các số còn lại, nên phần tử đầu `p[l]` là số **lớn nhất** trong `k` số đôi một khác nhau thuộc `{l+1, l+2, ...}` ⇒ `p[l] ≥ l + k = r + 1 = ans[l]`.

- Nếu `p[l] > ans[l]`: `ans` nhỏ hơn `p` ⇒ xong.
- Nếu `p[l] = r + 1`: `k` số trong khối đều `≤ r + 1` và khác nhau, tức chính là `{l+1, ..., r+1}`; do giảm dần nên khối của `p` **trùng khối của `ans`**; tiếp tục với khối sau.

Vậy `ans` là hoán vị hợp lệ nhỏ nhất. Cách 4 (ngăn xếp) và Cách 5 (công thức) sinh đúng cùng kết quả: ngăn xếp đảo ngược đúng các số từ `l+1` tới `r+1` của mỗi khối; công thức gán trực tiếp `r+1` rồi giảm dần. Ta đối chiếu bằng assert trong mục 12.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space (không tính kết quả) |
|---|---|---|
| 1. Mọi hoán vị | `O(n · (n+1)!)` | `O(n)` |
| 2. Quay lui | xấu nhất `O((n+1)!)` | `O(n)` ngăn xếp đệ quy |
| 3. Đảo đoạn | `O(n)` | `O(1)` |
| 4. Ngăn xếp | `O(n)` | `O(n)` |
| 5. Công thức `d` | `O(n)` | `O(n)` |

**Vì sao Cách 3 là `O(n)` dù có vòng `while` lồng nhau?** Vòng `while j` tăng `j` qua các chữ `D`, rồi `i` nhảy tới `j`; mỗi chữ `D` chỉ bị `j` đi qua **một lần**. Tổng bước của `j` là `O(n)`; tổng độ dài các đoạn đảo cũng `≤ n + 1`.

In [17]:
import time
n = 100000
for ten, s in [("toàn D", "D" * n), ("toàn I", "I" * n), ("DI xen kẽ", "DI" * (n // 2))]:
    t0 = time.perf_counter()
    a = dao_doan(s)
    b = ngan_xep(s)
    c = cong_thuc(s)
    assert a == b == c
    print(f"{ten:10}: {time.perf_counter() - t0:.3f}s cho 3 cách")

toàn D    : 0.079s cho 3 cách


toàn I    : 0.109s cho 3 cách


DI xen kẽ : 0.108s cho 3 cách


## 12. Bộ test

Chiến lược: (1) ví dụ đề; (2) biên (`"I"`, `"D"`, toàn `I`, toàn `D`); (3) **đối chiếu oracle vét cạn** với **mọi** chuỗi độ dài ≤ 7; (4) đối chiếu 3 cách tuyến tính trên chuỗi dài ngẫu nhiên (seed cố định), kèm **hàm kiểm tra hợp lệ**; (5) một tính chất của đáp án nhỏ nhất.

In [18]:
import random
from itertools import product

CACH = {"dao_doan": dao_doan, "ngan_xep": ngan_xep, "cong_thuc": cong_thuc}

# 1. Ví dụ của đề và biên
tests = [("I", [1, 2]), ("DI", [2, 1, 3]), ("D", [2, 1]), ("II", [1, 2, 3]), ("DD", [3, 2, 1]),
         ("ID", [1, 3, 2]), ("DDI", [3, 2, 1, 4]), ("IDID", [1, 3, 2, 5, 4])]
for s, kq in tests:
    for ten, f in CACH.items():
        assert f(s) == kq, (ten, s, f(s), kq)
    assert Solution().findPermutation(s) == kq
print("ví dụ + biên: OK")

ví dụ + biên: OK


In [19]:
# 2. Oracle vét cạn trên MỌI chuỗi độ dài 1..7 (254 chuỗi)
dem = 0
for L in range(1, 8):
    for tup in product("ID", repeat=L):
        s = "".join(tup)
        kq_dung = vet_can(s)
        assert hop_le(s, kq_dung)
        for ten, f in CACH.items():
            assert f(s) == kq_dung, (ten, s)
        dem += 1
print("khớp oracle vét cạn trên", dem, "chuỗi: OK")

# Quay lui (Cách 2) chỉ kiểm với độ dài <= 6
for L in range(1, 7):
    for tup in product("ID", repeat=L):
        s = "".join(tup)
        assert quay_lui(s) == vet_can(s)
print("quay lui khớp: OK")

khớp oracle vét cạn trên 254 chuỗi: OK
quay lui khớp: OK


In [20]:
# 3. Ngẫu nhiên có seed, độ dài lớn: kiểm tra hợp lệ + đối chiếu các cách
random.seed(484)
for _ in range(40):
    L = random.randint(1, 5000)
    s = "".join(random.choice("ID") for _ in range(L))
    a, b, c = dao_doan(s), ngan_xep(s), cong_thuc(s)
    assert a == b == c
    assert hop_le(s, a)
print("ngẫu nhiên độ dài tới 5000: OK")

# 4. Tính chất của đáp án nhỏ nhất: p[0] = 1 + số chữ D liền đầu chuỗi
for _ in range(200):
    L = random.randint(1, 60)
    s = "".join(random.choice("ID") for _ in range(L))
    so_D_dau = len(s) - len(s.lstrip("D"))
    assert dao_doan(s)[0] == 1 + so_D_dau
print("p[0] = 1 + (số D đầu chuỗi): OK")

ngẫu nhiên độ dài tới 5000: OK
p[0] = 1 + (số D đầu chuỗi): OK


**Phản ví dụ** cho ý nghĩ "mọi hoán vị hợp lệ đều đủ": với `s = "DI"` thì `[3, 1, 2]` hợp lệ nhưng **không** phải nhỏ nhất:

In [21]:
print(hop_le("DI", [3, 1, 2]), [3, 1, 2] > [2, 1, 3])     # True True: hợp lệ nhưng lớn hơn đáp án
assert hop_le("DI", [3, 1, 2]) and dao_doan("DI") == [2, 1, 3]

True True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng cách tham lam kiểu "hoán vị bất kỳ" (hợp lệ nhưng không nhỏ nhất)

**Code sai:** hai con trỏ `lo = 1`, `hi = n + 1`; gặp `I` đặt `lo` (rồi `lo += 1`), gặp `D` đặt `hi` (rồi `hi -= 1`). Đây là lời giải của bài "DI String Match" (chỉ cần **một** hoán vị hợp lệ).

**Vì sao sai ở bài này:** bài này đòi **nhỏ nhất theo từ điển**, mà cách trên đặt số **lớn nhất** vào chỗ `D`, nên phần tử đầu quá lớn.

In [22]:
def sai_tham_lam(s):
    lo, hi = 1, len(s) + 1
    res = []
    for ch in s:
        if ch == "I":
            res.append(lo); lo += 1
        else:
            res.append(hi); hi -= 1
    res.append(lo)
    return res

for s in ("DI", "DDI", "IDID"):
    print(s, sai_tham_lam(s), "hợp lệ:", hop_le(s, sai_tham_lam(s)), "| đúng:", dao_doan(s))
assert sai_tham_lam("DI") != dao_doan("DI") and hop_le("DI", sai_tham_lam("DI"))

DI [3, 1, 2] hợp lệ: True | đúng: [2, 1, 3]
DDI [4, 3, 1, 2] hợp lệ: True | đúng: [3, 2, 1, 4]
IDID [1, 5, 2, 4, 3] hợp lệ: True | đúng: [1, 3, 2, 5, 4]


**Code đúng:** đảo các đoạn `D` (Cách 3) hoặc ngăn xếp (Cách 4).

### Lỗi 2 — Đảo thiếu một phần tử ở cuối đoạn (lệch một)

**Code sai:** `ans[i:j] = ans[i:j][::-1]` (thay vì `j + 1`). Chuỗi `DD` phủ **3** phần tử (vị trí `i..j` với `j = i + 2`), nhưng cắt lát `[i:j]` chỉ lấy 2.

In [23]:
def sai_off_by_one(s):
    n = len(s)
    ans = list(range(1, n + 2))
    i = 0
    while i < n:
        j = i
        while j < n and s[j] == "D":
            j += 1
        ans[i:j] = ans[i:j][::-1]            # SAI: phải là j + 1
        i = max(i + 1, j)
    return ans

print(sai_off_by_one("DDI"), "hợp lệ:", hop_le("DDI", sai_off_by_one("DDI")))
print(dao_doan("DDI"))
assert not hop_le("DDI", sai_off_by_one("DDI"))

[2, 1, 3, 4] hợp lệ: False
[3, 2, 1, 4]


### Lỗi 3 — Quên đổ ngăn xếp ở cuối (hoặc sai độ dài kết quả)

**Code sai:** chạy `for i in range(1, n + 1)` (thiếu số `n+1`) hoặc không đổ khi `i == n + 1`. Kết quả hoán vị thiếu phần tử.

In [24]:
def sai_ngan_xep(s):
    n = len(s)
    res, st = [], []
    for i in range(1, n + 1):                 # SAI: phải là n + 2 (đến n + 1)
        st.append(i)
        if s[i - 1] == "I":
            while st:
                res.append(st.pop())
    return res

print(sai_ngan_xep("DDI"), "độ dài", len(sai_ngan_xep("DDI")), "nhưng cần", len("DDI") + 1)
print(sai_ngan_xep("DD"))                      # [] : không có chữ I nào nên chưa đổ gì
assert len(sai_ngan_xep("DDI")) != 4

[3, 2, 1] độ dài 3 nhưng cần 4
[]


**Code đúng:** `range(1, n + 2)` và đổ khi `i == n + 1 or s[i-1] == 'I'`.

### Lỗi 4 — Quên rằng đáp án có độ dài `n + 1`

Nhiều bạn khởi tạo `ans = list(range(1, n + 1))` (độ dài `n`). Hãy luôn tự kiểm tra bằng ví dụ nhỏ: `s = "I"` có độ dài 1 nhưng đáp án `[1, 2]` có độ dài 2.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** chuỗi `I`/`D` (tăng/giảm) và hỏi hoán vị nhỏ nhất / lớn nhất ⇒ nghĩ ngay tới **đảo đoạn `D`** hoặc **ngăn xếp**.
- **Ngăn xếp = đảo đoạn:** "dồn rồi đổ" tạo ra thứ tự ngược; khi cần đoạn giảm dần mà số được cấp theo thứ tự tăng, ngăn xếp rất hợp.
- **Nhỏ nhất theo từ điển** thường đi với **tham lam theo khối**: dùng các số nhỏ nhất còn lại cho khối đầu.
- **Hoán vị bất kỳ** (không cần nhỏ nhất) ⇒ hai con trỏ `lo/hi` (bài "DI String Match").
- Khi đề có nhiều đáp án đúng, hãy viết **hàm kiểm tra** (`hop_le`) trước; còn khi đáp án duy nhất như bài này, thêm **oracle vét cạn** cho input nhỏ.

## 15. Biến thể

1. **Hoán vị lớn nhất theo từ điển:** dùng phép đối xứng `v -> n + 2 - v` (đổi chiều mọi so sánh): lớn nhất của `s` = `n + 2 - (nhỏ nhất của s với I↔D đổi chỗ)`.
2. **DI String Match (942):** chỉ cần một hoán vị bất kỳ của `0..n` hợp lệ ⇒ `lo/hi`.
3. **Construct Smallest Number From DI String (2375):** dựng số nhỏ nhất gồm các chữ số **`1-9`**, không lặp, thỏa mẫu `I`/`D` (độ dài ≤ 8) ⇒ đúng bài này, ghép các số thành chuỗi chữ số.
4. **Đếm số hoán vị** thỏa chuỗi `I`/`D` (bài DP "Valid Permutations for DI Sequence", 903).
5. **Kiểm tra** hai chuỗi có cùng hoán vị nhỏ nhất không, v.v.

In [25]:
def lon_nhat(s):
    n = len(s)
    dao = "".join("D" if ch == "I" else "I" for ch in s)
    return [n + 2 - v for v in dao_doan(dao)]

def so_nho_nhat_2375(pattern):                 # biến thể: chữ số 1-9
    return "".join(map(str, dao_doan(pattern)))

print(lon_nhat("DI"), lon_nhat("IDID"))
print(so_nho_nhat_2375("IIIDIDDD"), so_nho_nhat_2375("DDD"))   # 123549876 4321
# kiểm tra biến thể lớn nhất bằng vét cạn
for L in range(1, 7):
    for tup in product("ID", repeat=L):
        s = "".join(tup)
        ds = [list(p) for p in permutations(range(1, L + 2)) if hop_le(s, p)]
        assert max(ds) == lon_nhat(s)
print("biến thể lớn nhất khớp vét cạn")

[3, 1, 2] [4, 5, 2, 3, 1]
123549876 4321


biến thể lớn nhất khớp vét cạn


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi hoán vị | sinh theo thứ tự từ điển, lấy cái hợp lệ đầu tiên | `O(n·(n+1)!)` | `O(n)` | oracle cho `n ≲ 7` |
| 2. Quay lui | dựng từng phần tử, thử từ nhỏ đến lớn | xấu nhất giai thừa | `O(n)` | minh họa cắt tỉa; không nộp |
| 3. Đảo đoạn `D` | bắt đầu `1..n+1`, đảo mỗi đoạn `D` | `O(n)` | `O(1)` thêm | cách nộp gọn nhất |
| 4. Ngăn xếp | dồn khi `D`, đổ khi `I` | `O(n)` | `O(n)` | dễ chứng minh, tổng quát hóa tốt |
| 5. Công thức `d` | tính trực tiếp từng giá trị | `O(n)` | `O(n)` | khi cần giá trị tại vị trí bất kỳ theo `d` |

**Khuyên dùng khi thi:** Cách 3 hoặc Cách 4.

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán kết quả của `dao_doan("DDID")` và `ngan_xep("IIDD")`; sau đó chạy để kiểm tra.

In [26]:
# Bài 1: ghi dự đoán của bạn
du_doan_1 = None
du_doan_2 = None
print("Thực tế:", dao_doan("DDID"), ngan_xep("IIDD"))

Thực tế: [3, 2, 1, 5, 4] [1, 2, 5, 4, 3]


**Bài 2 (tính tay).** Với `s = "IDDID"` (hoán vị 6 phần tử), hãy chia thành các khối, viết hoán vị nhỏ nhất, rồi kiểm tra bằng tay mỗi cặp liền kề khớp `I`/`D`. Sau đó đối chiếu bằng `vet_can`.

**Bài 3 (điền TODO).** Hoàn thiện hàm `dem_D_lien_tiep_todo(s)` trả về mảng `d` (độ dài `n + 1`, `d[n] = 0`) với `d[i]` là số `D` liên tiếp bắt đầu tại `s[i]`.

In [27]:
def dem_D_lien_tiep_todo(s):
    # TODO: tạo d = [0] * (len(s) + 1); duyệt i từ len(s)-1 về 0;
    #       nếu s[i] == 'D' thì d[i] = d[i+1] + 1, ngược lại d[i] = 0
    pass

kq_bai3 = None       # sau khi viết xong: kq_bai3 = dem_D_lien_tiep_todo("DDIDI")

**Bài 4 (tự viết lại).** Không nhìn các cách ở trên, tự viết `tim_hoan_vi(s)` bằng **ngăn xếp**. Phải trả đúng cho cả `"I"`, `"DD"` và `"IDID"`.

In [28]:
def tim_hoan_vi(s):
    # TODO: tự viết bằng ngăn xếp
    pass

**Bài 5 (biến thể / mini challenge).** Viết `lon_nhat_tu_viet(s)` trả về hoán vị **lớn nhất** theo thứ tự từ điển của `1..n+1` thỏa `s`. Gợi ý: đối xứng, hoặc "bắt đầu từ dãy giảm `n+1..1` rồi đảo các đoạn `I`".

In [29]:
def lon_nhat_tu_viet(s):
    # TODO: viết tự do
    pass

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):**
  - Bài 1: bắt đầu từ `[1,2,3,4,5]` (với `"DDID"`) rồi đảo từng đoạn `D`; ngăn xếp: nhớ đổ khi gặp `I`.
  - Bài 2: cắt khối sau mỗi chữ `I`: `IDDID` thành `I | DDI | D`.
  - Bài 3: quét từ phải sang trái.
- **Mức 2 (vừa):**
  - Bài 2: khối `I` phủ vị trí `0..0`; khối `DDI` phủ vị trí `1..3`; khối `D` (cuối chuỗi) phủ vị trí `4..5`. Khối phủ `l..r` dùng các số `l+1..r+1` theo thứ tự giảm dần.
  - Bài 4: `for i in range(1, n + 2): st.append(i); if i == n + 1 or s[i-1] == 'I': đổ`.
  - Bài 5: đảo `I`↔`D` của `s`, giải bài nhỏ nhất, rồi ánh xạ `v -> n + 2 - v`.
- **Mức 3 (gần đáp án):**
  - Bài 2: giá trị khối đầu `[1]`; khối `DDI` dùng các số `2..4` giảm: `4 3 2`; khối `D` dùng `5..6`: `6 5` ⇒ `[1, 4, 3, 2, 6, 5]`.
  - Bài 3: `d[i] = d[i+1] + 1 if s[i] == 'D' else 0`.
  - Bài 5: `[n + 2 - v for v in dao_doan(s_dao)]`.

## 19. Đáp án tham khảo cho bài tập

Mỗi đáp án được kiểm tra bằng `assert`.

In [30]:
# Bài 1
assert dao_doan("DDID") == [3, 2, 1, 5, 4]
assert ngan_xep("IIDD") == [1, 2, 5, 4, 3]
print(dao_doan("DDID"), ngan_xep("IIDD"))

# Bài 2
assert dao_doan("IDDID") == [1, 4, 3, 2, 6, 5]
assert vet_can("IDDID") == [1, 4, 3, 2, 6, 5]
assert hop_le("IDDID", [1, 4, 3, 2, 6, 5])

[3, 2, 1, 5, 4] [1, 2, 5, 4, 3]


In [31]:
# Bài 3
def dem_D_lien_tiep_dap_an(s):
    d = [0] * (len(s) + 1)
    for i in range(len(s) - 1, -1, -1):
        d[i] = d[i + 1] + 1 if s[i] == "D" else 0
    return d

assert dem_D_lien_tiep_dap_an("DDIDI") == [2, 1, 0, 1, 0, 0]
assert dem_D_lien_tiep_dap_an("I") == [0, 0]

# Bài 4
def tim_hoan_vi_dap_an(s):
    n = len(s)
    res, st = [], []
    for i in range(1, n + 2):
        st.append(i)
        if i == n + 1 or s[i - 1] == "I":
            while st:
                res.append(st.pop())
    return res

for s in ("I", "DD", "IDID"):
    assert tim_hoan_vi_dap_an(s) == vet_can(s)
for L in range(1, 7):
    for tup in product("ID", repeat=L):
        s = "".join(tup)
        assert tim_hoan_vi_dap_an(s) == vet_can(s)

In [32]:
# Bài 5
def lon_nhat_dap_an(s):
    n = len(s)
    s_dao = "".join("D" if ch == "I" else "I" for ch in s)
    return [n + 2 - v for v in dao_doan(s_dao)]

for L in range(1, 7):
    for tup in product("ID", repeat=L):
        s = "".join(tup)
        ds = [list(p) for p in permutations(range(1, L + 2)) if hop_le(s, p)]
        assert max(ds) == lon_nhat_dap_an(s)
print("Tất cả đáp án OK")

Tất cả đáp án OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (đảo đoạn `D`):**

```python
class Solution:
    def findPermutation(self, s: str) -> List[int]:
        n = len(s)
        ans = list(range(1, n + 2))        # dãy tăng 1..n+1 (độ dài n + 1)
        i = 0
        while i < n:
            j = i
            while j < n and s[j] == 'D':   # tìm hết chuỗi D liên tiếp
                j += 1
            ans[i:j + 1] = ans[i:j + 1][::-1]   # đảo cả đầu mút j
            i = max(i + 1, j)
        return ans
```

**Template (ngăn xếp):** `for i in range(1, n + 2): st.append(i); if i == n + 1 or s[i-1] == 'I': đổ hết st`.

**Checklist trước khi submit:**

- [ ] Kết quả có độ dài `n + 1`, gồm đúng các số `1..n+1`.
- [ ] Đảo đoạn `[i : j + 1]` (có `+ 1`).
- [ ] `i = max(i + 1, j)` để không bị kẹt vòng lặp khi không có `D`.
- [ ] Ngăn xếp: đổ khi `i == n + 1` **hoặc** `s[i-1] == 'I'`.
- [ ] Không dùng cách `lo/hi` (đó là hoán vị bất kỳ, không nhỏ nhất).
- [ ] Đã thử `"I"`, `"D"`, `"DI"`, `"DDI"`, toàn `D`, toàn `I`.
- [ ] Độ phức tạp `O(n)` thời gian, `O(n)` bộ nhớ.